# LASSI-Verify: Emperical Verification

**Argo Gateway API** Jupyter Notebook version

+ ***Post LASSI-Judge version with data collection for pass@k and energy_reduction@k***
+ No translation
+ Source code scripts from:
https://github.com/zjin-lcf/HeCBench/tree/master/src

NVIDIA GPU power measurements: https://pypi.org/project/nvidia-ml-py/  (pynvml)

Added support for AMD GPU device measurements and HIP code (ROCm)

In [1]:
from lassi.gpu_detector import detect_gpu_vendors

# Detect available GPUs
vendors = detect_gpu_vendors()
if not vendors:
    print("❌ No GPUs detected! LASSI requires GPU hardware.")
    raise SystemExit("Pipeline stopped - no GPU hardware found")

detected_vendor = vendors[0]['vendor']
gpu_count = vendors[0]['count']
print(f"🔍 Detected: {detected_vendor.upper()} with {gpu_count} GPU(s)")

# Vendor-specific initialization
if detected_vendor == "nvidia":
    # NVIDIA-specific setup
    import pynvml
    pynvml.nvmlInit()
    print("NVIDIA Driver Version:", pynvml.nvmlSystemGetDriverVersion())
    pynvml.nvmlShutdown()
    
    # Show NVIDIA system info
    !nvidia-smi
    !nvcc --version
    
elif detected_vendor == "amd":
    # AMD-specific setup
    from lassi.amd_power import amd_report_gpu_memory
    
    # Show AMD system info
    !rocminfo | head -20
    !rocm-smi
    !hipcc --version
    
print(f"\n✅ {detected_vendor.upper()} GPU environment ready for LASSI!")

🔍 Detected: NVIDIA with 1 GPU(s)
NVIDIA Driver Version: 570.211.01
Wed Mar 18 22:49:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 5000 Ada Gene...    Off |   00000000:01:00.0 Off |                  Off |
| N/A   41C    P8              3W /   80W |      14MiB /  16376MiB |      0%      Default |
|                                         |                        |     

## Setup

In [2]:
import os
import json
import csv
import time
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

# Import LASSI modules
import lassi.llm_config
# import lassi.pipeline # The Pipeline script calls the other LASSI libraries not imported here.
import lassi.pipeline_v4
from lassi.format_colors import color
from lassi.prompt_dictionary import PromptDictionary
from lassi.monitor import start_monitor, log_figure

# Enable the autoreload extension
%load_ext autoreload
%reload_ext autoreload

# Reload all modules imported with %aimport every time before executing subsequent code.
%autoreload 1

# We need to reload the libraries while under development
%aimport lassi.format_colors
%aimport lassi.prompt_dictionary
%aimport lassi.llm_config
%aimport lassi.llm_methods
%aimport lassi.pipeline_v4
%aimport lassi.get_power
%aimport lassi.argostream_curl
%aimport lassi.code_manager
%aimport lassi.pipeline_methods
%aimport lassi.verify
%aimport lassi.monitor

In [3]:
# ── LASSI Monitor — live output dashboard ──────────────────────────────────────
# Starts a local web server and streams all pipeline print() output to a browser
# tab in real-time.  Run this cell once after the imports cell above.
# Then open the link below in a new browser tab and keep it visible.
#
# If matplotlib figures are produced (EE task), they appear automatically in a
# sidebar panel on the right side of the dashboard — no extra calls needed.
#
# The dashboard accumulates the full session history, so a late-opened tab
# will still see everything printed since start_monitor() was called.
#
# SSH remote access: forward port 5007 to your local machine, then open
#   http://localhost:5007  in your local browser.
start_monitor()   # default port 5007 — pass port=XXXX to change

'http://localhost:5007'

## Code Generation Experiment - Setup + Run

_____
***STEP 1/2: Input experimental configuration parameters in the following method.***

In [4]:
def experimental_setup():
    # Load config file with source application names and input parameters
    with open("source_apps.json", "r") as f:
        source_apps = json.load(f)
    
    """
    [1] ENTER APPLICATION FOLDER NAME FROM HeCBench
        >> Execution input parameters provided.
    
    *** USE FOR LASSI-Verfify experiment: ***
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
     (this one may show a lot of failures)

    HeCBench apps:
    -- matrix-rotate ["30000", "100"]  ["10000", "1"]  # Usage: %s <matrix size> <repeat>
    -- jacobi [""] (none required)
    -- layout ["2500"] ["1"]
    -- atomicCost ["16", "10"] ["1"]
    -- segment-reduce ["16384", "100"] (a replacement for atomicCost, same "Data compression and reduction" category in HeCBench)
       --> (may be trivial for LASSI-Verify)
    -- dense-embedding ["100000", "128", "100"] ["10000", "8", "1"]
    -- pathfinder ["100000", "10000", "2500"] ["100000", "1000", "5"] ["10000", "1000", "1000"]  # Usage: <column length> <row length> <pyramid_height>
    -- bsearch ["8388608", "1000"] ["10000", "1"]  # Usage: <number of elements> <repeat>
    -- entropy ["10000", "1024", "1"] ["8192", "8192", "500"]   # Usage: <width> <height> <repeat>
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
    -- randomAccess ["10"] ["1"]  # Usage: <repeat>
    -- lid-driven-cavity [""]
    -- murmurhash3 ["750000, 500"]
    -- chacha20 ["300000"]
    -- jaccard ["1024, 512, 1000"]
    -- keogh ["256", "22500000", "100"]
    -- floydwarshall ["2048", "100", "16"]
    -- extrema ["750"]
    -- threadfence ["100", "250000000"]
    -- all-pairs-distance ["10000"]
    -- marchingCubes ["100"]

    Altis apps:
    -- level1/bfs ["--size 3"]
    -- level2/raytracing [""]
    -- level2/mandelbrot ["--imageSize 1024 --iterations 100000"]

    Miniapps:
    -- miniMDock ["-lfile", "./input/7cpa/7cpa_ligand.pdbqt", "-nrun 500"]
    -- XSBench ["-m", "event", "-l", "1700000000"]
    
    """

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # Enter source application to process through LASSI
    source_collection = "HeCBench"  # "Miniapps" "HeCBench"
    source_app = "pathfinder"  # "miniMDock" jacobi
    make_script = False  # Used only for Miniapps
    build_script = False  # False (for HeCBench) Use for Altis only
    

    """
    Valid source code folder names (under source_codes):
    1. HeCBench
    2. altis/src/cuda
    3. miniMDock/bin/
    """
    source_folder = source_apps[source_collection]["source_folder"]  # "altis/src/cuda/level2"    #  "HeCBench" "altis/src/cuda/level1"
    app_folder_name = source_apps[source_collection][source_app]["app_folder"]
    lines_of_code = source_apps[source_collection][source_app]["code_lines"]
    if make_script:
        app_name = source_app  # app_folder_name
        file_name = "Main"  # kernels
    else:
        app_name = "mt_" + app_folder_name + "-cuda_main"
        file_name = ""

    
    """
    [2] SELECT THE CODE TRANSLATION MAPPING DIRECTION EXPERIMENT CONFIGURATION
    
    CUDA -- OMP -- HIP -- SYCL -- FORTRAN
    """   
    # These should be the same if *not* doing translation.
    source_lang = "CUDA"
    target_lang = "CUDA"

    verifyprompt = True  # True or False -- Print out LLM prompt in stdout to preview
    
    # TOGGLE USE CODE KNOWLEDGE CONTEXT FOR TARGET
    use_context_summaries_only = True  # True or False
    few_shot_learning = False  # True or False
    
    # LLM judge code analysis (pre-cursor to LASSI-Verify)
    run_code_analysis = False  # True or False
    
    # THESE WILL NO LONGER BE OPTIONS
    # test_code_first = True  # True or False
    use_context = False  # True or False -- NOT USED IN v2, v3
    use_self_prompting = False  # True or False -- NOT USED IN v2, v3
    baseline_oneshot_goal = True  # True or False -- NOT USED IN v2, v3
    
    # -----------------------------------------------------------------------
    # TASK MODE CONFIGURATION
    # Set these two flags together depending on the task:
    #
    #   ENERGY EFFICIENCY (EE) task:
    #     power_measure    = True   -- measure GPU power/energy during execution
    #     yes_power_feedback = True -- run LASSI self-improvement loop using energy as the optimization metric
    #
    #   RUNTIME PERFORMANCE task:
    #     power_measure    = False  -- skip power measurement, use wall-clock duration instead
    #     yes_power_feedback = True -- run LASSI self-improvement loop using runtime as the optimization metric
    #
    #   CODE TRANSLATION task (one-shot, no iterative improvement):
    #     power_measure    = False  -- no measurement needed
    #     yes_power_feedback = False -- skip feedback loop entirely (generate once, compile, execute, judge, done)
    #     NOTE: translation mode still has a known infinite-loop-on-success issue when yes_power_feedback=False
    # -----------------------------------------------------------------------
    power_measure = False  # True (EE task) | False (runtime or translation task)
    vary_llm_temp = True  # True or False

    # TOGGLE LASSI SELF-IMPROVEMENT FEEDBACK LOOP
    yes_power_feedback = True  # True (EE or runtime task) | False (translation task, one-shot only)

    # -----------------------------------------------------------------------
    # LASSI-VERIFY TIER 2 CONFIGURATION
    # run_tier2=True: insert passthrough observation blocks and validate empirical equivalence
    # tier2_tolerance: relative floating-point tolerance for passthrough comparison (default 1e-6)
    # -----------------------------------------------------------------------
    run_tier2 = True         # True to enable Tier 2 empirical equivalence checking
    tier2_tolerance = 1e-6    # Relative tolerance for passthrough block comparison

    # LASSI-VERIFY TIER 3 CONFIGURATION
    # run_tier3=True: generate metamorphic contract and run variant filter (requires run_tier2=True)
    # tier3_tolerance: relative tolerance for metamorphic relation checks (default 1e-5, slightly looser than Tier 2)
    # -----------------------------------------------------------------------
    run_tier3 = True        # True to enable Tier 3 metamorphic testing
    tier3_tolerance = 1e-5   # Relative tolerance for metamorphic relation comparison

    # -----------------------------------------------------------------------
    # LLM-AS-A-JUDGE CONFIGURATION
    # run_llm_judge=True: enable LLM functional equivalence judge (pre-Tier-1/2 stop-gap)
    # run_llm_judge=False: pure three-tier mode — Tier 1 + Tier 2 determine validity only
    # -----------------------------------------------------------------------
    run_llm_judge = True     # True to enable LLM judge; False for pure three-tier LASSI-Verify mode

    # ASSIGN CODE COMPILE COMMAND
    code_build_script = ""
    code_compiler = ""
    code_compiler_kwds = ""
    if source_lang == "CUDA":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=CUDA DEVICE=GPU CARD=NVIDIA NUMWI=64"
            elif source_app == "XSBench":
                code_compiler = "cuda/make"
        else:
            code_compiler = "nvcc"
            code_compiler_kwds = "-std=c++14 -Xcompiler -Wall -arch=sm_89 -O3"
            # -arch=sm_70 for aiops01 GPU (Argonne)  # -arch=sm_80 for A100s
            # -arch=sm_60 for P100
            # -arch=sm_89 for RTX 5000
        if yes_power_feedback:
            context_knowledge_selection = "cuda_optimization_summary"
            context_fewshot_examples = "cuda_c_few_shot_examples"
        else:
            context_knowledge_selection = "cuda"
    elif source_lang == "OMP":
        # For A100 on Linux_x86_64 or Linux_aarch64
        # code_compiler = "g++" # "/opt/nvidia/hpc_sdk/Linux_aarch64/2024/compilers/bin/nvc++"  # 
        # code_compiler_kwds = "-fopenmp -O3" # "-Wall -O3 -Minfo -mp=gpu -gpu=cc80"  # -c
        # !! TODO - REVIEW: code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc80"  # "-fopenmp -O3 -c"
        # For RTX 5000 Ada on Linux x86_64:
        code_compiler = "/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++" # "g++"
        code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc89"  # "-fopenmp -O3"
        context_knowledge_selection = "omp"
    elif source_lang == "HIP":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=HIP DEVICE=GPU CARD=AMD NUMWI=64 HIP_PATH=/opt/rocm-6.3.3"
            elif source_app == "XSBench":
                code_compiler = "hip/make"
            else:
                code_compiler_kwds = ""
        else:
            code_compiler = "hipcc"
            code_compiler_kwds = "-std=c++14 -Wall -O3 --amdgpu-target=gfx908"
        if yes_power_feedback:
            context_knowledge_selection = "hip_optimization_summary"
            context_fewshot_examples = "hip_c_few_shot_examples"
        else:
            context_knowledge_selection = "hip"  # TODO

    # TOGGLE EXECUTE PIPELINE
    yes_execute = True  # True / False
    if yes_execute:
        # Enter execution input parameters, if needed
        # A list of parameter string values, e.g., ["10000", "1
        # ****************************
        # ---> EXPERIMENT UPDATE FIELD - Now, automated:
        exe_input_parameters = source_apps[source_collection][source_app]["app_params"]
    else:
        exe_input_parameters = [""]

    
    """
    [3] SELECT THE LLM TO USE FOR THE CODE TRANSLATION EXPERIMENT
    """
    # SELECT A MODEL

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # -- FROM ARGO:
    # model = "gpt35"  # (max token = 4096)
    # model = "gpt35large"  # (max token = 16384)
    # model = "gpt4large"  # (max token = 32768)
    # model = "gpt4turbo"  # (max token = 128000)
    # model = "gpto4mini"  # (max token = 128000)
    # model_coder = "gpto4mini"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4o"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 200000)
    # model_coder = "gpt4olatest"  # (max token = 200000)
    model = "gpt54"
    model_coder = "gpt54"
    model_judge = "gpt41"  # (max token = 200000)

    # -- FROM CEREBRAS
    # model = "gpt-oss-120b"
    # model_judge = "gpt-oss-120b"
    # model_coder = "qwen-3-coder-480b"
    
    # -- FROM OLLAMA
    # model = "devstral:24b" # (max token = 128000)
    # model_coder = "devstral:24b" # (max token = 128000)
    # model_judge = "devstral:24b" # (max token = 128000)
    # model = "qwq:latest"  # (max token = 128000)
    # model_coder = "qwen2.5-coder:32b"  # (max token = 32768)
    # model_judge = "qwq:latest"  # (max token = 128000)
    # model = "qwq:latest"  # (max token = 131072)
    # model = "gemma3:27b"  # crashing on code gen
    # model_coder = "gemma3:27b"
    # model = "granite3.2:8b-instruct-q8_0"
    # model_coder = "granite-code:34b-instruct-q8_0"
    # model = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model = "dolphin3"  # (max token = 131072)
    # model = "llama3.3:latest"  # (max token = 131072)
    # model = "phi4" # (max token = 16384)
    # model_coder = "codestral:22b"  # (max token = 32768)
    # model = "codestral:22b-v0.1-q8_0"  # (max token = 32768)
    # model = "wizardcoder:33b-v1.1-q8_0"  # (max token = 16384)
    # model_coder = "deepseek-coder:33b-instruct-q8_0"  # (max token = 16384)
    # model = "deepseek-coder-v2:16b-lite-instruct-fp16"  # (max token = 128000)
    # model = "deepseek-coder-v2:16b-lite-instruct-q8_0"
    # model = "phi3:14b"  # (max token = 131072)
    # model_judge = "llama3.3:70b"  # (max token = 128000)
    # model = "qwen3:latest"
    # model_coder = "qwen3:latest"
    # model_judge = "qwen3:latest"  # (max token = 40960)
    # model_coder = "qwen3-coder:30b"  # context 262144
    # model = "gpt-oss:20b"
    # model_judge = "gpt-oss:20b"
  
    # Set maximum context token count size for selected LLM:
    num_ctx = 200000
    num_ctx_coder = 200000

    # Needed only for Argo endpoints
    uri = "local"  # local  remote
    lassi.llm_config.argo_username = "mdearing"  # Argo Gateway username

    # ASSIGN LLM API SOURCE:
    if model in ["gpt54", "gpt5", "gpt35", "gpt35large", "gpt4", "gpt4large", "gpt4turbo", "gpt4o", "gpt4olatest", "gpt41",
                 "gpto1preview", "gpto1mini", "gpto3mini", "gpto4mini", "claudesonnet4", "claudesonnet46"]:
        llm_source = "argo"
    elif model in ["gpt-oss-120b", "qwen-3-coder-480b"]:
        llm_source = "cerebras"
    elif ("GGUF" in model and "hf.co" not in model):
        llm_source = "lmstudio"
    else:
        llm_source = "ollama"

    """
    [4] SET ADDITIONAL PIPELINE PARAMETERS
    """
    experiment = "TEST_FULLEXP_verify_0318-2026_" + model_coder + "_" + app_folder_name + "_runtm_" + detected_vendor.upper()  # Assign any label for experiment name.
    display_response = True
    test_with_error = False
    generate_test_case = False

    # Unload any previously loaded LLMs in the Ollama server 
    # (clear out the memory)
    if llm_source == "ollama":
        lassi.llm_config.unload_ollama_model(model)
        lassi.llm_config.unload_ollama_model(model_coder)

    """
    [5] SET DATA COLLECTION CONFIGURATION
    """
    # Data spreadsheet column headers
    DATAFIELDNAMES = [
        "trial_number",
        "timestamp",
        "experiment",
        "gpu_vendor",
        "app_name",
        "code_line_count",
        "llm",
        "llm_coder",
        "llm_judge",
        "source_v0_energy_J",
        "source_v0_avg_power_W",
        "source_v0_runtime_s",
        "source_verify_runtime_s",  # ★ instrumented source runtime
        "base_v1_energy_J",
        "base_v1_avg_power_W",
        "base_v1_runtime_s",
        "base_v1_valid",
        "base_v1_energy_diff",
        "base_v1_avg_power_diff",
        "base_v1_runtime_diff",
        "zeroshot_runtime_s",        # ★ schema-aligned alias for base_v1_runtime_s
        "zeroshot_compile_status",   # ★ "success" | "compile_error" | "skipped"
        "zeroshot_execute_status",   # ★ "success" | "execute_error" | "skipped"
        # "base_v1_optimization_methods",
        # "base_v2_energy_J",
        # "base_v2_avg_power_W",
        # "base_v2_runtime_s",
        # "base_v2_valid",
        # "base_v2_selfcorrections",
        # "base_v2_energy_diff",
        # "base_v2_avg_power_diff",
        # "base_v2_runtime_diff",
        # "base_v2_optimization_methods",
        "lassiee_energy_J",
        "lassiee_avg_power_W",
        "lassiee_runtime_s",
        "lassiee_valid",
        "lassiee_version",
        "lassiee_llm_temp",
        "lassiee_selfcorrections",
        "lassiee_eeiterations",
        "lassiee_energy_diff",
        "lassiee_avg_power_diff",
        "lassiee_runtime_diff",
        "lassiee_optimization_methods",
        "total_selfcorrections",
        "compile_selfcorrections",   # ★
        "execute_selfcorrections",   # ★
        "tier1_selfcorrections",     # ★
        "tier2_selfcorrections",     # ★
        "tier3_selfcorrections",     # ★
        "global_counter_at_end",     # ★ global verify counter at trial end
        "total_code_versions",       # ★ distinct LASSI code block indices
        "total_code_attempts",       # ★ rows in attempt log for this trial
        "total_eeiterations",
        "pipeline_runtime_min",
        "tier2_pass_count",
        "tier2_fail_total",
        "tier3_pass_count",
        "tier3_fail_total",
        "tier3_active_relations",
        "tier3_inactive_relations",  # ★
        "judge_false_positives",
        "judge_false_negatives",
        "lassiee_clean_status",
        "lassiee_clean_runtime_s",
        "lassiee_clean_runtime_diff",
        "lassiee_clean_energy_J",
        "lassiee_clean_energy_diff",
        "base_llm_tier1_verdict",
        "base_llm_tier2_verdict",
        "base_llm_tier3_verdict",
        "base_llm_verified_valid"
    ]
    
    return (app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder, llm_source, use_context, use_context_summaries_only,
            few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler, code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters,
            experiment, display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal, use_self_prompting, vary_llm_temp,
            verifyprompt, DATAFIELDNAMES, detected_vendor.upper(),
            run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
            run_llm_judge)

# EXPERIMENTAL DATA COLLETION MANAGEMENT
def init_data_collection(dir_path, DATAFIELDNAMES, filename="result_lassi-verify_pipeline_data.csv"):
    os.makedirs(dir_path, exist_ok=True)
    fp = os.path.join(dir_path, filename)
    if not os.path.exists(fp):
        with open(fp, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=DATAFIELDNAMES)
            writer.writeheader()
    return fp

_____
***STEP 2/2: Run the following block to implement the pipeline with the configuration parameters.***

In [5]:
# Experimental repeat for pass@k metrics
trial_number = 1
total_trials = 3

# Load prompt dictionary (to capture recent manual changes to script, if any)
prompts = PromptDictionary()

all_trials_start = time.time()

# Store the current home working directory (needed for code execution)
original_home_directory = os.getcwd()


# COLLECT THE EXPERIMENT CONFIGURATIONS
(
    app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder,
    llm_source, use_context, use_context_summaries_only, few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler,
    code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters, experiment,
    display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal,
    use_self_prompting, vary_llm_temp, verifyprompt, DATAFIELDNAMES, gpu_vendor,
    run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
    run_llm_judge
) = experimental_setup()

# IMPORT CONTEXT KNOWLEDGE FOR TARGET PROGRAMMING LANGUAGE
if gpu_vendor == "NVIDIA":
    input_context = "[BACKGROUND KNOWLEDGE: CUDA C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("cuda") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: CUDA C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]
elif gpu_vendor == "AMD":
    input_context = "[BACKGROUND KNOWLEDGE: HIP C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("hip") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: HIP C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]


# DEFINE PROMPT TEMPLATES
# TODO: MOST TASK PROMPT TEMPLATES WILL BE HANDLED BY THE PIPELINE METHODS
# If we are doing the runtime task, then set the system prompts, otherwise they are for ee
# we need to bring back in the translation systems prompt reference when we create LASSI-X
if (power_measure == False and yes_power_feedback == True):
    system_prompt_selection_index = "general_system_" + source_lang + "_runtime"
else:
    system_prompt_selection_index = source_lang + "_ee"  # "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_system_prompt = prompts.get_system_prompt(str(system_prompt_selection_index))

coderefactor_prompt_selection_index = source_lang  # + "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
if (not baseline_oneshot_goal and use_self_prompting):
    # TODO: THIS SHOULD BE A DEAD BRANCH BECAUSE WE DON'T WANT TO USE baseline_oneshot_goal ANYMORE
    his_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index))
else: 
    if (power_measure == False and yes_power_feedback == True):
        # RUNTIME TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_runtime"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext_runtime"))
    else:
        # EE TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext"))

# LOAD INPUT CODE TO REFACTOR
input_code_folder = source_folder + "/" + app_folder_name + "/"
if build_script:
    input_code_filename = app_name
elif make_script:
    input_code_filename = file_name
else:
    input_code_filename = app_folder_name + "-" + source_lang.lower() + "_main"
if source_lang == "OMP" or source_lang == "SYCL":
    input_code_extension = "cpp"
elif source_lang == "CUDA" or source_lang == "HIP":
    if (make_script and source_lang == "HIP"):
        input_code_extension = "cpp"  # custom for miniMDock and XSBench
    else:
        input_code_extension = "cu"
if make_script:
    source_file_directory = "generated_codes/" + input_code_folder
else:
    source_file_directory = "source_codes/" + input_code_folder
if make_script:
    # SELECT FILE TO REFACTOR from miniapp
    if source_lang == "CUDA":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/cuda/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    elif source_lang == "HIP":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/hip/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_file_subdirectory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
else:
    input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
with open(source_file_path, 'r') as file:
    input_code = file.read()
file.close()


# SPECIFY OUTPUT CODE SAVE
if build_script:
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
    output_code_file_name = app_name
elif make_script:
    output_code_file_name = file_name
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
else:
    output_code_directory = "generated_codes/" + llm_source + "_" + model.replace('/', '--') + "/promptset_" + str(coderefactor_prompt_selection_index)
    output_code_file_name = input_code_filename + "_ee_refactorcode_" + source_lang + "_exp" + experiment

now = datetime.now(ZoneInfo("America/Chicago"))
dt_string = now.strftime("%m-%d-%Y_t%H_%M_%S")

# SPECIFIY DATA COLLECTION SAVE
experiment_data_directory = "generated_codes/experiment/" + experiment + "__" + dt_string
# Convert to absolute directory::
experiment_data_directory = os.path.abspath(experiment_data_directory)
experiment_data_file_object = init_data_collection(experiment_data_directory, DATAFIELDNAMES)
experiment_data = dict.fromkeys(DATAFIELDNAMES, None) # all keys start out as None
# Table 2 ★ per-attempt log — one row per code generation attempt (including self-correction retries)
attempt_log_file_path = init_data_collection(
    experiment_data_directory, lassi.pipeline_v4.ATTEMPT_LOG_FIELDNAMES,
    filename="result_lassi-verify_attempt_log.csv"
)
experiment_data["gpu_vendor"] = str(detected_vendor.upper())

while trial_number <= total_trials:
    # For the miniApp, we need to reset the source kernel file
    #  because the pipeline for this scenario must overwrite the 
    #  source code in the generated_codes folder.
    if make_script:
        if source_lang == "CUDA":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels_orig_merged.cu"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main_orig_merged.cu"
        elif source_lang == "HIP":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels_orig_merged.cpp"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main_orig_merged.cpp"
        try:
            print(f"[reset] Copying clean kernel -> working file\n"
                  f"        src: {clean_kernel_file}\n"
                  f"        dst: {working_kernel_file}")
            shutil.copyfile(clean_kernel_file, working_kernel_file)
            print("[reset] Success: working file reset to clean copy.")
        except FileNotFoundError as e:
            print(f"[reset] ERROR: file not found: {e.filename}")
        except PermissionError as e:
            print(f"[reset] ERROR: permission denied: {e.filename}")
        except OSError as e:
            # catches other IO errors (disk full, cross-device issues, etc.)
            print(f"[reset] ERROR: {e.__class__.__name__}: {e}")

    # CREATE METADATA RECORD CONTENT
    meta_data_file = "***** EXPERIMENT CONFIGURATION SUMMARY *****\n"
    meta_data_file += "\nGPU Vendor: " + str(detected_vendor.upper())
    meta_data_file += "\nDate-time: " + dt_string
    meta_data_file += "\nApplication: " + app_folder_name
    meta_data_file += "\n\nRefactoring task: " + system_prompt_selection_index
    meta_data_file += "\n\nLLM Source: " + llm_source
    meta_data_file += "\nModel: " + model
    meta_data_file += "\nModel coder: " + model_coder
    meta_data_file += "\nModel judge: " + model_judge
    meta_data_file += "\nSystem Prompt Index: " + str(coderefactor_prompt_selection_index)
    meta_data_file += "\nCode file template: " + output_code_directory + "/" + output_code_file_name
    meta_data_file += "\n\nLLM System Prompt: " + "\n\n" + this_llm_system_prompt
    # meta_data_file += "\n\nLLM Prompt for Code Generation: " + "\n\n" + this_llm_refactoredcode_prompt
    meta_data_file += "\n\nPower measurement: " + str(power_measure)
    
    # Display metadata
    print("\n" + meta_data_file + "\n")
    
    print("\n\n" + color.BLUE + color.BOLD + "***** BEGIN THE LASSI ENERGY EFFICIENCY PIPELINE (v4) *****" + color.END + " Trial #: " + str(trial_number) + "\n\n")
    
    # RUN LASSI-EE PIPELINE
    complete = lassi.pipeline_v4.auto_code_llmgeneration_pipeline(
        prompts,
        this_llm_system_prompt,
        this_llm_refactoredcode_prompt,
        llm_source, 
        model,
        model_coder,
        model_judge,
        num_ctx,
        num_ctx_coder,
        # arrayormatrix_size_range,
        # thread_count_range,
        use_context,
        use_context_summaries_only,
        few_shot_learning,
        run_code_analysis,
        input_context,
        input_code_file_subdirectory,
        input_code,
        lines_of_code,
        # target_code,
        target_lang,
        input_code_extension,
        source_lang,
        code_compiler,
        code_compiler_kwds,
        build_script,
        make_script,
        code_build_script,
        output_code_directory,
        output_code_file_name,
        source_file_path,
        display_response,
        test_with_error,
        generate_test_case,
        yes_execute,
        exe_input_parameters,
        meta_data_file,
        power_measure,
        yes_power_feedback,
        dt_string,
        baseline_oneshot_goal,
        use_self_prompting,
        vary_llm_temp,
        experiment,
        verifyprompt,
        trial_number,
        experiment_data_directory,
        experiment_data_file_object,
        experiment_data,
        app_name,
        run_tier2=run_tier2,
        tier2_tolerance=tier2_tolerance,
        run_tier3=run_tier3,
        tier3_tolerance=tier3_tolerance,
        run_llm_judge=run_llm_judge,
        attempt_log_file_path=attempt_log_file_path
    )
    
    # Reset to original working directory
    os.chdir(original_home_directory)
    
    if complete:
        print("\n\n" + color.GREEN + color.BOLD + "***** LASSI PIPELINE COMPLETED *****\n\n" + color.END)
    else:
        print("\n\n" + color.RED + color.BOLD + "***** LASSI PIPELINE FAILED TO COMPLETE *****\n\n" + color.END)

    # Increment experiment counter
    trial_number = trial_number + 1


all_trials_end = time.time() - all_trials_start

print("\n\n" + color.PURPLE + color.BOLD + "***** ALL TRIALS COMPLETED ***** Total time (min): " + str(round(all_trials_end/60, 2)) + "\n\n" + color.END)




***** EXPERIMENT CONFIGURATION SUMMARY *****

GPU Vendor: NVIDIA
Date-time: 03-18-2026_t22_49_39
Application: pathfinder

Refactoring task: general_system_CUDA_runtime

LLM Source: argo
Model: gpt54
Model coder: gpt54
Model judge: gpt41
System Prompt Index: CUDA
Code file template: generated_codes/argo_gpt54/promptset_CUDA/pathfinder-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA

LLM System Prompt: 

You are an expert C++/CUDA engineer. Your goal is to refactor provided CUDA C++ code to reduce wall-clock execution time by improving GPU and CPU utilization and parallelism, while preserving all original functionality and expected output of the provided code. You must also carefully read and incorporate the best practices and CUDA optimization techniques from the provided background documents and refactoring plans. You always surround your generated code with the three characters ```.

Power measurement: False



***** BEGIN THE LASSI ENERG

Waiting for the system to settle down for 1 seconds...: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/pathfinder/pathfinder-cuda_main
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/pathfinder
executable:  pathfinder-cuda_main
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/pathfinder
Execute command: ['./pathfinder-cuda_main', '100000', '10000', '2500']

running: executable -  pathfinder-cuda_main

----- < BEGIN: STANDARD OUTPUT FROM CODE EXECUTION > -----

Total kernel execution time: 16.929045 (s)
Device offloading time = 17.546587 (s)

THE RETURN CODE:  0

----- < END: Standard Output > -----


Program executed successfully!

----- END OF STAGE 0: TEST SOURCE -----

None

*** DEBUG: code_version: 0 | tracking_best_version: 0 | code_blocks len: 1 (next idx: 1) ***

----- STARTING CODE INSERT - Invariant Assertions -----


=====> VERIFY THE PROMPTING <===== [tokens: 16,754]

TE

```cpp
/***********************************************************************
 * PathFinder uses dynamic programming to find a path on a 2-D grid from
 * the bottom row to the top row with the smallest accumulated weights,
 * where each step of the path moves straight ahead or diagonally ahead.
 * It iterates row by row, each node picks a neighboring node in the
 * previous row that has the smallest accumulated weight, and adds its
 * own weight to the sum.
 *
 * This kernel uses the technique of ghost zone optimization
 ***********************************************************************/

// Other header files.
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <assert.h>
#include <iostream>
#include <sys/time.h>
#include <cuda.h>
#include <cstdio>
#include <cmath>


// halo width along one direction when advancing to the next iteration
#define HALO     1
#define STR_SIZE 256
#define DEVICE   0
#define M_SEED   9
#define IN_RANGE(x, min, max)  ((x)>=(min) && (x)<=(max))
#define CLAMP_RANGE(x, min, max) x = (x<(min)) ? min : ((x>(max)) ? max : x )
#define MIN(a, b) ((a)<=(b) ? (a) : (b))


void fatal(char *s)
{
  fprintf(stderr, "error: %s\n", s);
}

double get_time() {
  struct timeval t;
  gettimeofday(&t,NULL);
  return t.tv_sec+t.tv_usec*1e-6;
}

__global__ void pathfinder (
    const int*__restrict__ gpuWall,
    const int*__restrict__ gpuSrc,
          int*__restrict__ gpuResult,
          int*__restrict__ outputBuffer,
    const int iteration,
    const int theHalo,
    const int borderCols,
    const int cols,
    const int t)
{
  int BLOCK_SIZE = blockDim.x;
  int bx = blockIdx.x;
  int tx = threadIdx.x;
  __shared__ int prev[250];
  __shared__ int result[250];

  // Each block finally computes result for a small block
  // after N iterations.
  // it is the non-overlapping small blocks that cover
  // all the input data

  // calculate the small block size.
  int small_block_cols = BLOCK_SIZE - (iteration*theHalo*2);

  // calculate the boundary for the block according to
  // the boundary of its small block
  int blkX = (small_block_cols*bx) - borderCols;
  int blkXmax = blkX+BLOCK_SIZE-1;

  // calculate the global thread coordination
  int xidx = blkX+tx;

  // effective range within this block that falls within
  // the valid range of the input data
  // used to rule out computation outside the boundary.
  int validXmin = (blkX < 0) ? -blkX : 0;
  int validXmax = (blkXmax > cols-1) ? BLOCK_SIZE-1-(blkXmax-cols+1) : BLOCK_SIZE-1;

  int W = tx-1;
  int E = tx+1;

  W = (W < validXmin) ? validXmin : W;
  E = (E > validXmax) ? validXmax : E;

  bool isValid = IN_RANGE(tx, validXmin, validXmax);

  if(IN_RANGE(xidx, 0, cols-1))
  {
    prev[tx] = gpuSrc[xidx];
  }

  __syncthreads();

  bool computed;
  for (int i = 0; i < iteration; i++)
  {
    computed = false;

    if( IN_RANGE(tx, i+1, BLOCK_SIZE-i-2) && isValid )
    {
      computed = true;
      int left = prev[W];
      int up = prev[tx];
      int right = prev[E];
      int shortest = MIN(left, up);
      shortest = MIN(shortest, right);

      int index = cols*(t+i)+xidx;
      result[tx] = shortest + gpuWall[index];

      // ===================================================================
      // add debugging info to the debug output buffer...
      if (tx==11 && i==0)
      {
        // set bufIndex to what value/range of values you want to know.
        int bufIndex = gpuSrc[xidx];
        // dont touch the line below.
        outputBuffer[bufIndex] = 1;
      }
      // ===================================================================
    }

    __syncthreads();

    if(i==iteration-1)
    {
      // we are on the last iteration, and thus don't need to 
      // compute for the next step.
      break;
    }

    if(computed)
    {
      //Assign the computation range
      prev[tx] = result[tx];
    }
    __syncthreads();
  }

  // update the global memory
  // after the last iteration, only threads coordinated within the
  // small block perform the calculation and switch on "computed"
  if (computed)
  {
    gpuResult[xidx] = result[tx];
  }
}

int main(int argc, char** argv)
{
  // Program variables.
  int   rows, cols;
  int*  data;
  int** wall;
  int*  result;
  int   pyramid_height;

  if (argc == 4)
  {
    cols = atoi(argv[1]);
    rows = atoi(argv[2]);
    pyramid_height = atoi(argv[3]);
  }
  else
  {
    printf("Usage: %s <column length> <row length> <pyramid_height>\n", argv[0]);

    exit(0);
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_cli_dimensions_and_configuration
  {
    bool _t1_pass = (argc == 4 && cols > 0 && rows > 0 && pyramid_height > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_cli_dims_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  data = new int[rows * cols];
  wall = new int*[rows];
  for (int n = 0; n < rows; n++)
  {
    // wall[n] is set to be the nth row of the data array.
    wall[n] = data + cols * n;
  }
  result = new int[cols];

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_host_allocations
  {
    bool _t1_pass = (data != NULL && wall != NULL && result != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_allocs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  int seed = M_SEED;
  srand(seed);

  for (int i = 0; i < rows; i++)
  {
    for (int j = 0; j < cols; j++)
    {
      wall[i][j] = rand() % 10;
    }
  }
#ifdef BENCH_PRINT
  for (int i = 0; i < rows; i++)
  {
    for (int j = 0; j < cols; j++)
    {
      printf("%d ", wall[i][j]);
    }
    printf("\n");
  }
#endif

  // Pyramid parameters.
  const int borderCols = (pyramid_height) * HALO;

  /* printf("pyramidHeight: %d\ngridSize: [%d]\nborder:[%d]\nblockSize: %d\nblockGrid:[%d]\ntargetBlock:[%d]\n",
     pyramid_height, cols, borderCols, NUMBER_THREADS, blockCols, smallBlockCol); */

  int size = rows * cols; // the size (global work size) is a multiple of lws 

  // running the opencl application shows lws=4000 (cpu) and lws=250 (gpu)
  int lws = 250;
  int* outputBuffer = (int*)calloc(16384, sizeof(int));
  int theHalo = HALO;

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_pretransfer_host_data_and_runtime_parameters
  {
    bool _t1_pass = (outputBuffer != NULL && size >= cols && cols > 0 && rows > 0 &&
                     lws > 0 && pyramid_height > 0 && borderCols >= 0 && theHalo == HALO);
    if (_t1_pass)
    {
      for (int i = 0; i < rows && _t1_pass; i++)
      {
        for (int j = 0; j < cols; j++)
        {
          int _t1_v = wall[i][j];
          if (_t1_v < 0 || _t1_v > 9)
          {
            _t1_pass = false;
            break;
          }
        }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_pre_htod_inputs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  double offload_start = get_time();

  int* d_gpuWall;
  cudaMalloc((void**)&d_gpuWall, sizeof(int)*(size-cols));
  cudaMemcpy(d_gpuWall, data+cols, sizeof(int)*(size-cols), cudaMemcpyHostToDevice);

  int* d_gpuSrc;
  cudaMalloc((void**)&d_gpuSrc, sizeof(int)*cols);
  cudaMemcpy(d_gpuSrc, data, sizeof(int)*cols, cudaMemcpyHostToDevice);

  int* d_gpuResult;
  cudaMalloc((void**)&d_gpuResult, sizeof(int)*cols);

  int* d_outputBuffer;
  cudaMalloc((void**)&d_outputBuffer, sizeof(int)*16384);

  dim3 gridDim (size/lws);
  dim3 blockDim (lws);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_kernel_launch_configuration
  {
    bool _t1_pass = (gridDim.x > 0 && blockDim.x > 0 && blockDim.x <= 1024 && lws == 250);
    printf("LASSI-VERIFY ASSERTIONS [main_kernel_launch_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaDeviceSynchronize();
  double kstart = get_time();

  for (int t = 0; t < rows - 1; t += pyramid_height)
  {
    // Calculate this for the kernel argument...
    int iteration = MIN(pyramid_height, rows-t-1);

    pathfinder<<<gridDim, blockDim>>>(
        d_gpuWall, d_gpuSrc, d_gpuResult, d_outputBuffer, 
        iteration, theHalo, borderCols, cols, t);

    int* temp = d_gpuResult;
    d_gpuResult = d_gpuSrc;
    d_gpuSrc = temp;
  }

  cudaDeviceSynchronize();
  double kend = get_time();
  printf("Total kernel execution time: %lf (s)\n", kend - kstart);

  cudaMemcpy(result, d_gpuSrc, sizeof(int)*cols, cudaMemcpyDeviceToHost);
  cudaMemcpy(outputBuffer, d_outputBuffer, sizeof(int)*16348, cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_posttransfer_result_and_debug_buffer
  {
    bool _t1_pass = (result != NULL && outputBuffer != NULL && cols > 0);
    if (_t1_pass)
    {
      for (int i = 0; i < cols; i++)
      {
        if (!std::isfinite((double)result[i]))
        {
          _t1_pass = false;
          break;
        }
      }
    }
    if (_t1_pass)
    {
      for (int i = 0; i < 16348; i++)
      {
        if (!(outputBuffer[i] == 0 || outputBuffer[i] == 1))
        {
          _t1_pass = false;
          break;
        }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_post_dtoh_outputs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaFree(d_gpuResult);
  cudaFree(d_gpuSrc);
  cudaFree(d_gpuWall);
  cudaFree(d_outputBuffer);

  double offload_end = get_time();
  printf("Device offloading time = %lf (s)\n", offload_end - offload_start);

  // add a null terminator at the end of the string.
  outputBuffer[16383] = '\0';

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_final_result_properties_and_sentinel_buffer_state
  {
    bool _t1_pass = (outputBuffer[16383] == '\0');
    if (_t1_pass)
    {
      for (int i = 0; i < cols; i++)
      {
        if (result[i] < 0)
        {
          _t1_pass = false;
          break;
        }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_exit_result_properties_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

#ifdef BENCH_PRINT
  for (int i = 0; i < cols; i++)
    printf("%d ", data[i]);
  printf("\n");
  for (int i = 0; i < cols; i++)
    printf("%d ", result[i]);
  printf("\n");
#endif

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // Memory cleanup here.
  delete[] data;
  delete[] wall;
  delete[] result;
  free(outputBuffer);

  return EXIT_SUCCESS;
}
```


----- STARTING CODE INSERT - Passthrough Data Dump Blocks -----


=====> VERIFY THE PROMPTING <===== [tokens: 9,812]

TEMPERATURE: 0.2

SYSTEM: You are an expert C++/CUDA engineer performing surgical code insertion. Your ONLY task is to INSERT new instrumentation code at specified locations in an existing source file. You MUST preserve ALL existing code EXACTLY as written — do not modify, simplify, reorder, optimize, or remove any existing code, comments, blank lines, or whitespace. Every single line of the input file must appear in your output, in the same order. You always surround your generated code with the three characters ```.

PROMPT: 
        You are preparing GPU-accelerated scientific code for Tier 2 empirical equivalence checking
        as part of the LASSI-Verify verification framework.

        Your task is to instrument the source code with **passthrough data blocks** at key observation
        points. These blocks capture computational state to stdout in a structured,

```cpp
/***********************************************************************
 * PathFinder uses dynamic programming to find a path on a 2-D grid from
 * the bottom row to the top row with the smallest accumulated weights,
 * where each step of the path moves straight ahead or diagonally ahead.
 * It iterates row by row, each node picks a neighboring node in the
 * previous row that has the smallest accumulated weight, and adds its
 * own weight to the sum.
 *
 * This kernel uses the technique of ghost zone optimization
 ***********************************************************************/

// Other header files.
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <assert.h>
#include <iostream>
#include <sys/time.h>
#include <cuda.h>
#include <cstdio>
#include <cmath>


// halo width along one direction when advancing to the next iteration
#define HALO     1
#define STR_SIZE 256
#define DEVICE   0
#define M_SEED   9
#define IN_RANGE(x, min, max)  ((x)>=(min) && (x)<=(max))
#define CLAMP_RANGE(x, min, max) x = (x<(min)) ? min : ((x>(max)) ? max : x )
#define MIN(a, b) ((a)<=(b) ? (a) : (b))


void fatal(char *s)
{
  fprintf(stderr, "error: %s\n", s);
}

double get_time() {
  struct timeval t;
  gettimeofday(&t,NULL);
  return t.tv_sec+t.tv_usec*1e-6;
}

__global__ void pathfinder (
    const int*__restrict__ gpuWall,
    const int*__restrict__ gpuSrc,
          int*__restrict__ gpuResult,
          int*__restrict__ outputBuffer,
    const int iteration,
    const int theHalo,
    const int borderCols,
    const int cols,
    const int t)
{
  int BLOCK_SIZE = blockDim.x;
  int bx = blockIdx.x;
  int tx = threadIdx.x;
  __shared__ int prev[250];
  __shared__ int result[250];

  // Each block finally computes result for a small block
  // after N iterations.
  // it is the non-overlapping small blocks that cover
  // all the input data

  // calculate the small block size.
  int small_block_cols = BLOCK_SIZE - (iteration*theHalo*2);

  // calculate the boundary for the block according to
  // the boundary of its small block
  int blkX = (small_block_cols*bx) - borderCols;
  int blkXmax = blkX+BLOCK_SIZE-1;

  // calculate the global thread coordination
  int xidx = blkX+tx;

  // effective range within this block that falls within
  // the valid range of the input data
  // used to rule out computation outside the boundary.
  int validXmin = (blkX < 0) ? -blkX : 0;
  int validXmax = (blkXmax > cols-1) ? BLOCK_SIZE-1-(blkXmax-cols+1) : BLOCK_SIZE-1;

  int W = tx-1;
  int E = tx+1;

  W = (W < validXmin) ? validXmin : W;
  E = (E > validXmax) ? validXmax : E;

  bool isValid = IN_RANGE(tx, validXmin, validXmax);

  if(IN_RANGE(xidx, 0, cols-1))
  {
    prev[tx] = gpuSrc[xidx];
  }

  __syncthreads();

  bool computed;
  for (int i = 0; i < iteration; i++)
  {
    computed = false;

    if( IN_RANGE(tx, i+1, BLOCK_SIZE-i-2) && isValid )
    {
      computed = true;
      int left = prev[W];
      int up = prev[tx];
      int right = prev[E];
      int shortest = MIN(left, up);
      shortest = MIN(shortest, right);

      int index = cols*(t+i)+xidx;
      result[tx] = shortest + gpuWall[index];

      // ===================================================================
      // add debugging info to the debug output buffer...
      if (tx==11 && i==0)
      {
        // set bufIndex to what value/range of values you want to know.
        int bufIndex = gpuSrc[xidx];
        // dont touch the line below.
        outputBuffer[bufIndex] = 1;
      }
      // ===================================================================
    }

    __syncthreads();

    if(i==iteration-1)
    {
      // we are on the last iteration, and thus don't need to 
      // compute for the next step.
      break;
    }

    if(computed)
    {
      //Assign the computation range
      prev[tx] = result[tx];
    }
    __syncthreads();
  }

  // update the global memory
  // after the last iteration, only threads coordinated within the
  // small block perform the calculation and switch on "computed"
  if (computed)
  {
    gpuResult[xidx] = result[tx];
  }
}

int main(int argc, char** argv)
{
  // Program variables.
  int   rows, cols;
  int*  data;
  int** wall;
  int*  result;
  int   pyramid_height;

  if (argc == 4)
  {
    cols = atoi(argv[1]);
    rows = atoi(argv[2]);
    pyramid_height = atoi(argv[3]);
  }
  else
  {
    printf("Usage: %s <column length> <row length> <pyramid_height>\n", argv[0]);

    exit(0);
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_cli_dimensions_and_configuration
  {
    bool _t1_pass = (argc == 4 && cols > 0 && rows > 0 && pyramid_height > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_cli_dims_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  data = new int[rows * cols];
  wall = new int*[rows];
  for (int n = 0; n < rows; n++)
  {
    // wall[n] is set to be the nth row of the data array.
    wall[n] = data + cols * n;
  }
  result = new int[cols];

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_host_allocations
  {
    bool _t1_pass = (data != NULL && wall != NULL && result != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_allocs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  int seed = M_SEED;
  srand(seed);

  for (int i = 0; i < rows; i++)
  {
    for (int j = 0; j < cols; j++)
    {
      wall[i][j] = rand() % 10;
    }
  }
#ifdef BENCH_PRINT
  for (int i = 0; i < rows; i++)
  {
    for (int j = 0; j < cols; j++)
    {
      printf("%d ", wall[i][j]);
    }
    printf("\n");
  }
#endif

  // Pyramid parameters.
  const int borderCols = (pyramid_height) * HALO;

  /* printf("pyramidHeight: %d\ngridSize: [%d]\nborder:[%d]\nblockSize: %d\nblockGrid:[%d]\ntargetBlock:[%d]\n",
     pyramid_height, cols, borderCols, NUMBER_THREADS, blockCols, smallBlockCol); */

  int size = rows * cols; // the size (global work size) is a multiple of lws 

  // running the opencl application shows lws=4000 (cpu) and lws=250 (gpu)
  int lws = 250;
  int* outputBuffer = (int*)calloc(16384, sizeof(int));
  int theHalo = HALO;

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_pretransfer_host_data_and_runtime_parameters
  {
    bool _t1_pass = (outputBuffer != NULL && size >= cols && cols > 0 && rows > 0 &&
                     lws > 0 && pyramid_height > 0 && borderCols >= 0 && theHalo == HALO);
    if (_t1_pass)
    {
      for (int i = 0; i < rows && _t1_pass; i++)
      {
        for (int j = 0; j < cols; j++)
        {
          int _t1_v = wall[i][j];
          if (_t1_v < 0 || _t1_v > 9)
          {
            _t1_pass = false;
            break;
          }
        }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_pre_htod_inputs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  double offload_start = get_time();

  int* d_gpuWall;
  cudaMalloc((void**)&d_gpuWall, sizeof(int)*(size-cols));
  cudaMemcpy(d_gpuWall, data+cols, sizeof(int)*(size-cols), cudaMemcpyHostToDevice);

  int* d_gpuSrc;
  cudaMalloc((void**)&d_gpuSrc, sizeof(int)*cols);
  cudaMemcpy(d_gpuSrc, data, sizeof(int)*cols, cudaMemcpyHostToDevice);

  int* d_gpuResult;
  cudaMalloc((void**)&d_gpuResult, sizeof(int)*cols);

  int* d_outputBuffer;
  cudaMalloc((void**)&d_outputBuffer, sizeof(int)*16384);

  dim3 gridDim (size/lws);
  dim3 blockDim (lws);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_kernel_launch_configuration
  {
    bool _t1_pass = (gridDim.x > 0 && blockDim.x > 0 && blockDim.x <= 1024 && lws == 250);
    printf("LASSI-VERIFY ASSERTIONS [main_kernel_launch_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaDeviceSynchronize();
  double kstart = get_time();

  for (int t = 0; t < rows - 1; t += pyramid_height)
  {
    // Calculate this for the kernel argument...
    int iteration = MIN(pyramid_height, rows-t-1);

    pathfinder<<<gridDim, blockDim>>>(
        d_gpuWall, d_gpuSrc, d_gpuResult, d_outputBuffer, 
        iteration, theHalo, borderCols, cols, t);

    int* temp = d_gpuResult;
    d_gpuResult = d_gpuSrc;
    d_gpuSrc = temp;
  }

  cudaDeviceSynchronize();
  double kend = get_time();
  printf("Total kernel execution time: %lf (s)\n", kend - kstart);

  cudaMemcpy(result, d_gpuSrc, sizeof(int)*cols, cudaMemcpyDeviceToHost);
  cudaMemcpy(outputBuffer, d_outputBuffer, sizeof(int)*16348, cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_posttransfer_result_and_debug_buffer
  {
    bool _t1_pass = (result != NULL && outputBuffer != NULL && cols > 0);
    if (_t1_pass)
    {
      for (int i = 0; i < cols; i++)
      {
        if (!std::isfinite((double)result[i]))
        {
          _t1_pass = false;
          break;
        }
      }
    }
    if (_t1_pass)
    {
      for (int i = 0; i < 16348; i++)
      {
        if (!(outputBuffer[i] == 0 || outputBuffer[i] == 1))
        {
          _t1_pass = false;
          break;
        }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_post_dtoh_outputs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [final_result_buffer] (do not edit or delete this block) >>>
  {
      // --- LASSI-VERIFY PASSTHROUGH: capture final dynamic programming result buffer ---
      int* lv_host_buf = (int*) malloc(cols * sizeof(int));
      cudaMemcpy(lv_host_buf, d_gpuSrc, cols * sizeof(int), cudaMemcpyDeviceToHost);

      int lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
      double lv_sum = 0.0;
      for (int lv_i = 0; lv_i < cols; lv_i++) {
          if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
          if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
          lv_sum += (double)lv_host_buf[lv_i];
      }
      double lv_mean = lv_sum / cols;

      printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [final_result_buffer] ===\n");
      printf("Buffer: d_gpuSrc\n");
      printf("Type: int\n");
      printf("Elements: %d\n", (int)cols);
      printf("Statistics: min=%d, max=%d, mean=%.17g\n",
             lv_min, lv_max, lv_mean);

      int lv_sample = (cols < 10) ? (int)cols : 10;
      printf("Sample_first10:");
      for (int lv_i = 0; lv_i < lv_sample; lv_i++)
          printf(" %d", lv_host_buf[lv_i]);
      printf("\n");

      if (cols > 10) {
          printf("Sample_last10:");
          for (int lv_i = (int)cols - 10; lv_i < (int)cols; lv_i++)
              printf(" %d", lv_host_buf[lv_i]);
          printf("\n");
      }

      if ((int)cols > 10) {
          // Sample_spread10: 10 evenly-spaced interior elements at indices (k+1)*N/11.
          // Covers interior of the buffer regardless of array layout, catching bugs
          // that Sample_first10/last10 miss when boundary elements are unchanged.
          printf("Sample_spread10:");
          for (int lv_k = 0; lv_k < 10; lv_k++) {
              int lv_spread_idx = (lv_k + 1) * (int)cols / 11;
              printf(" %d", lv_host_buf[lv_spread_idx]);
          }
          printf("\n");
      }
      printf("=== END LASSI-VERIFY PASSTHROUGH [final_result_buffer] ===\n");
      free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [final_debug_output_buffer] (do not edit or delete this block) >>>
  {
      // --- LASSI-VERIFY PASSTHROUGH: capture final debug output buffer ---
      int* lv_host_buf = (int*) malloc(16384 * sizeof(int));
      cudaMemcpy(lv_host_buf, d_outputBuffer, 16384 * sizeof(int), cudaMemcpyDeviceToHost);

      int lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
      double lv_sum = 0.0;
      for (int lv_i = 0; lv_i < 16384; lv_i++) {
          if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
          if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
          lv_sum += (double)lv_host_buf[lv_i];
      }
      double lv_mean = lv_sum / 16384;

      printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [final_debug_output_buffer] ===\n");
      printf("Buffer: d_outputBuffer\n");
      printf("Type: int\n");
      printf("Elements: %d\n", (int)16384);
      printf("Statistics: min=%d, max=%d, mean=%.17g\n",
             lv_min, lv_max, lv_mean);

      int lv_sample = (16384 < 10) ? (int)16384 : 10;
      printf("Sample_first10:");
      for (int lv_i = 0; lv_i < lv_sample; lv_i++)
          printf(" %d", lv_host_buf[lv_i]);
      printf("\n");

      if (16384 > 10) {
          printf("Sample_last10:");
          for (int lv_i = (int)16384 - 10; lv_i < (int)16384; lv_i++)
              printf(" %d", lv_host_buf[lv_i]);
          printf("\n");
      }

      if ((int)16384 > 10) {
          // Sample_spread10: 10 evenly-spaced interior elements at indices (k+1)*N/11.
          // Covers interior of the buffer regardless of array layout, catching bugs
          // that Sample_first10/last10 miss when boundary elements are unchanged.
          printf("Sample_spread10:");
          for (int lv_k = 0; lv_k < 10; lv_k++) {
              int lv_spread_idx = (lv_k + 1) * (int)16384 / 11;
              printf(" %d", lv_host_buf[lv_spread_idx]);
          }
          printf("\n");
      }
      printf("=== END LASSI-VERIFY PASSTHROUGH [final_debug_output_buffer] ===\n");
      free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  cudaFree(d_gpuResult);
  cudaFree(d_gpuSrc);
  cudaFree(d_gpuWall);
  cudaFree(d_outputBuffer);

  double offload_end = get_time();
  printf("Device offloading time = %lf (s)\n", offload_end - offload_start);

  // add a null terminator at the end of the string.
  outputBuffer[16383] = '\0';

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - validate_final_result_properties_and_sentinel_buffer_state
  {
    bool _t1_pass = (outputBuffer[16383] == '\0');
    if (_t1_pass)
    {
      for (int i = 0; i < cols; i++)
      {
        if (result[i] < 0)
        {
          _t1_pass = false;
          break;
        }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_exit_result_properties_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

#ifdef BENCH_PRINT
  for (int i = 0; i < cols; i++)
    printf("%d ", data[i]);
  printf("\n");
  for (int i = 0; i < cols; i++)
    printf("%d ", result[i]);
  printf("\n");
#endif

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // Memory cleanup here.
  delete[] data;
  delete[] wall;
  delete[] result;
  free(outputBuffer);

  return EXIT_SUCCESS;
}
```


----- LASSI-Verify Tier 2: passthrough blocks inserted into SOURCE_verify -----

----- LASSI-Verify: instrumented source written to /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA__03-18-2026_t22_49_39/pathfinder-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA_verify_Trial1.cu -----


----- SOURCE CODE: Test compile and execute -----

Test compile the original source code: /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA__03-18-2026_t22_49_39/pathfinder-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA_verify_Trial1.cu
Compile command: ['nvcc', '-std=c++14', '-Xcompiler', '-Wall', '-arch=sm_89', '-O3', '/home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA__03-1

Waiting for the system to settle down for 1 seconds...: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA__03-18-2026_t22_49_39/pathfinder-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA_verify_Trial1
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA__03-18-2026_t22_49_39
executable:  pathfinder-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA_verify_Trial1
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA__03-18-2026_t22_49_39
Execute command: ['./pathfinder-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0318-2026_gpt54_pathfinder_runtm_NVIDIA_verify_Trial1', '100000', '10000', '2500']

running: exe

KeyboardInterrupt: 